# Combined Chemical → Disease Network (pathway middleman removed)

Replaces the pathway-enrichment-only GNEA/KEGG notebook. Pipeline:

```
chemical ──(KEGG pathway membership)──► pathway ──(disease's pathway profile)──► disease
                         ⇓  collapse on shared pathways  ⇓
                       chemical ───────────────► disease     (direct edge)
```

The two layers you already had — **pathway↔chemical** and **pathway↔disease** —
are merged on their shared pathways, then the pathway nodes are dropped, leaving a
direct **chemical → disease** graph. Edges that reproduce known CTD associations
validate the method; edges to chemicals *not* originally listed for a disease are
**novel candidates** — exactly the predictions that complement the ML model.

Nodes are labelled with **chemical common names**, not CIDs (the reverse of the
MetaboAnalyst name→ID step). KEGG's `list` endpoint provides the ID→name map, so
no separate API is needed.


In [ ]:
!pip install -q requests pandas networkx pyvis
import requests, time, re, json, os
import pandas as pd
from collections import defaultdict
KEGG='https://rest.kegg.jp'
def kget(path):
    for _ in range(3):
        try:
            r=requests.get(f'{KEGG}/{path}',timeout=30)
            if r.status_code==200: return r.text
        except Exception: time.sleep(1)
    return ''

In [ ]:
# Load the master CSV
from google.colab import drive; drive.mount('/content/drive')
CSV='/content/drive/MyDrive/AI Projects/Respiratory Diseases - Sheet1.csv'   # <-- adjust
df=pd.read_csv(CSV)
df.columns=[c.strip().lstrip('#').strip() for c in df.columns]
df['DiseaseName']=df['DiseaseName'].astype(str).str.strip()
df['ChemicalName']=df['ChemicalName'].astype(str).str.strip()
df=df.drop_duplicates(['DiseaseName','ChemicalName']).reset_index(drop=True)
print(df.shape,'| diseases:',df.DiseaseName.nunique(),'| chemicals:',df.ChemicalName.nunique())

In [ ]:
# 1. Map each chemical NAME -> KEGG compound ID (cached). Keep the common name.
CACHE='/content/drive/MyDrive/AI Projects/name_to_kegg.csv'
def name_to_kegg(name):
    # KEGG free-text compound search; take first hit
    txt=kget(f'find/compound/{requests.utils.quote(name)}')
    for line in txt.strip().split('\n'):
        if line.startswith('cpd:'):
            return line.split('\t')[0].replace('cpd:','')
    return None

names=sorted(df.ChemicalName.unique())
if os.path.exists(CACHE):
    m=pd.read_csv(CACHE)
else:
    rows=[]
    for i,n in enumerate(names):
        rows.append({'ChemicalName':n,'KEGG_ID':name_to_kegg(n)})
        if (i+1)%25==0: print(i+1,'/',len(names)); time.sleep(0.1)
    m=pd.DataFrame(rows); m.to_csv(CACHE,index=False)
m=m.dropna(subset=['KEGG_ID'])
print('mapped',len(m),'/',len(names),'chemicals to KEGG IDs')
name_of=dict(zip(m.KEGG_ID,m.ChemicalName))   # KEGG ID -> common name (reverse map)

In [ ]:
# 2. chemical -> pathways (KEGG), in batches of 25. Drop global/overview maps.
GLOBAL={'01100','01110','01120','01200','01210','01212','01220','01230','01232',
        '01240','01250','01060','01061','01062','01063','01064','01065','01070'}
ids=m.KEGG_ID.tolist()
cpd_paths=defaultdict(set)
for k in range(0,len(ids),25):
    batch='+'.join(ids[k:k+25])
    for line in kget(f'link/pathway/{batch}').strip().split('\n'):
        if '\t' not in line: continue
        c,p=line.split('\t'); p=p.replace('path:map','')
        if p not in GLOBAL:
            cpd_paths[c.replace('cpd:','')].add(p)
    time.sleep(0.1)
print('compounds with pathways:',len(cpd_paths))

In [ ]:
# 3. disease -> pathway profile, then collapse to direct chemical -> disease edges
cid_disease=defaultdict(set)
for _,r in df.merge(m,on='ChemicalName').iterrows():
    cid_disease[r.KEGG_ID].add(r.DiseaseName)
dis_paths=defaultdict(set)
for cid,ds in cid_disease.items():
    for d in ds: dis_paths[d]|=cpd_paths.get(cid,set())

known=set((d,c) for c,ds in cid_disease.items() for d in ds)
rows=[]
for cid,paths in cpd_paths.items():
    if not paths: continue
    for d,dp in dis_paths.items():
        sh=paths&dp
        if not sh: continue
        rows.append({'ChemicalName':name_of.get(cid,cid),'KEGG_ID':cid,'Disease':d,
                     'shared_pathways':len(sh),'jaccard':round(len(sh)/len(paths|dp),4),
                     'link_type':'known' if (d,cid) in known else 'novel_candidate'})
edges=pd.DataFrame(rows).sort_values(['Disease','shared_pathways'],ascending=[True,False])
OUT='/content/drive/MyDrive/AI Projects'
edges.to_csv(f'{OUT}/combined_chemical_disease_edges.csv',index=False)
print('edges:',len(edges),'| known',int((edges.link_type=='known').sum()),
      '| novel',int((edges.link_type=='novel_candidate').sum()))
edges[edges.link_type=='novel_candidate'].head(15)

In [ ]:
# 4. Interactive network with COMMON-NAME labels (chemical = circle, disease = box)
from pyvis.network import Network
net=Network(height='720px',width='100%',bgcolor='#ffffff',notebook=True,cdn_resources='in_line')
top=edges.sort_values('shared_pathways',ascending=False).head(150)
dcolor='#c0392b'
for d in top.Disease.unique():
    net.add_node('D:'+d,label=d,shape='box',color=dcolor,size=26)
for _,r in top.iterrows():
    nid='C:'+r.KEGG_ID
    net.add_node(nid,label=r.ChemicalName,shape='dot',
                 color='#2980b9' if r.link_type=='known' else '#27ae60',
                 size=10+2*r.shared_pathways,
                 title=f"{r.ChemicalName} ({r.KEGG_ID}) — {r.link_type}")
    net.add_edge(nid,'D:'+r.Disease,value=int(r.shared_pathways),
                 color='#95a5a6' if r.link_type=='known' else '#2ecc71',
                 title=f"{r.shared_pathways} shared pathways, Jaccard {r.jaccard}")
net.force_atlas_2based()
net.show(f'{OUT}/chemical_disease_network.html')
print('blue = known link, green = novel candidate; node size = shared pathways')

In [ ]:
# 5. Export for the FastAPI backend
json.dump({'diseases':sorted(dis_paths),
           'edges':edges.to_dict(orient='records'),
           'kegg_name':name_of},
          open(f'{OUT}/network_data.json','w'),indent=1)
print('wrote network_data.json for the backend')